In [1]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

# ---------------------- 1. 加载数据 ----------------------
# 使用sklearn自带的digits数据集（MNIST子集，8x8手写数字）
digits = load_digits()
X = digits.data  # 形状：(1797, 64)
y = digits.target.reshape(-1, 1)  # 标签：(1797, 1)

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---------------------- 2. 数据预处理 ----------------------
# 图像已经是展平的向量，这里只做标签one-hot编码
encoder = OneHotEncoder(sparse_output=False)
y_train_onehot = encoder.fit_transform(y_train)
y_test_onehot = encoder.transform(y_test)

# 归一化（像素值缩放到0~1）
X_train = X_train / 16.0
X_test = X_test / 16.0

# 定义参数
n_samples, n_features = X_train.shape
n_classes = y_train_onehot.shape[1]
lr = 0.1
batch_size = 32
epochs = 50

# 初始化权重和偏置
W = np.random.randn(n_features, n_classes) * 0.01
b = np.zeros((1, n_classes))

# ---------------------- 3. 实现softmax和交叉熵损失 ----------------------
def softmax(z):
    # 数值稳定版softmax
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

def cross_entropy_loss(y_pred, y_true):
    # 交叉熵损失
    m = y_pred.shape[0]
    log_likelihood = -np.log(y_pred[range(m), y_true.argmax(axis=1)])
    return np.mean(log_likelihood)

# ---------------------- 4. 小批量SGD训练 ----------------------
for epoch in range(epochs):
    # 打乱数据
    indices = np.random.permutation(n_samples)
    X_shuffled = X_train[indices]
    y_shuffled = y_train_onehot[indices]
    
    for i in range(0, n_samples, batch_size):
        # 取一个batch
        X_batch = X_shuffled[i:i+batch_size]
        y_batch = y_shuffled[i:i+batch_size]
        
        # 前向传播
        z = X_batch @ W + b
        y_pred = softmax(z)
        
        # 反向传播：计算梯度
        dz = y_pred - y_batch
        dW = (X_batch.T @ dz) / X_batch.shape[0]
        db = np.sum(dz, axis=0, keepdims=True) / X_batch.shape[0]
        
        # 更新参数
        W -= lr * dW
        b -= lr * db
    
    # 每个epoch打印损失
    z_train = X_train @ W + b
    y_train_pred = softmax(z_train)
    train_loss = cross_entropy_loss(y_train_pred, y_train_onehot)
    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}")

# ---------------------- 5. 测试集准确率 ----------------------
z_test = X_test @ W + b
y_test_pred = softmax(z_test)
y_pred_labels = np.argmax(y_test_pred, axis=1)
y_true_labels = np.argmax(y_test_onehot, axis=1)

accuracy = np.mean(y_pred_labels == y_true_labels)
print(f"\nTest Accuracy: {accuracy:.4f}")

Epoch 1/50, Train Loss: 1.6042
Epoch 2/50, Train Loss: 1.1888
Epoch 3/50, Train Loss: 0.9458
Epoch 4/50, Train Loss: 0.7888
Epoch 5/50, Train Loss: 0.6815
Epoch 6/50, Train Loss: 0.6049
Epoch 7/50, Train Loss: 0.5478
Epoch 8/50, Train Loss: 0.5036
Epoch 9/50, Train Loss: 0.4683
Epoch 10/50, Train Loss: 0.4384
Epoch 11/50, Train Loss: 0.4143
Epoch 12/50, Train Loss: 0.3936
Epoch 13/50, Train Loss: 0.3753
Epoch 14/50, Train Loss: 0.3590
Epoch 15/50, Train Loss: 0.3453
Epoch 16/50, Train Loss: 0.3328
Epoch 17/50, Train Loss: 0.3216
Epoch 18/50, Train Loss: 0.3121
Epoch 19/50, Train Loss: 0.3023
Epoch 20/50, Train Loss: 0.2940
Epoch 21/50, Train Loss: 0.2861
Epoch 22/50, Train Loss: 0.2789
Epoch 23/50, Train Loss: 0.2725
Epoch 24/50, Train Loss: 0.2665
Epoch 25/50, Train Loss: 0.2605
Epoch 26/50, Train Loss: 0.2554
Epoch 27/50, Train Loss: 0.2505
Epoch 28/50, Train Loss: 0.2457
Epoch 29/50, Train Loss: 0.2410
Epoch 30/50, Train Loss: 0.2370
Epoch 31/50, Train Loss: 0.2328
Epoch 32/50, Trai